# Prophet — Walmart Store Sales Forecasting (theory-focused)

MLflow experiment: `Prophet_Training`.
Runs: `Prophet_SubsetSelection`, `Prophet_Yearly`, `Prophet_Holidays`, `Prophet_Final`.

Prophet fits **one model per series**, which does not scale to ~3,000 (Store, Dept) series,
so per `EXPERIMENTS.md` / `TASKS.md` (T16) this notebook is deliberately a **subset demo**
used for the theory writeup, not a full submission model. Metric: WMAE, holiday weeks
weighted 5x, computed only over the subset's validation weeks (last 39 weeks of history).

## Environment

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install mlflow dagshub kaggle prophet pandas pyarrow python-dotenv joblib
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

    from google.colab import userdata
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        f.write(userdata.get("KAGGLE_JSON"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    if not os.path.exists("data/train.csv"):
        !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
        import zipfile, glob
        os.chdir("data")
        while glob.glob("*.zip"):
            for z in glob.glob("*.zip"):
                with zipfile.ZipFile(z) as zf:
                    zf.extractall(".")
                os.remove(z)
        os.chdir("..")
print("IN_COLAB =", IN_COLAB)

In [ ]:
import sys
from pathlib import Path

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.isdir("ML-final"):
        !git clone https://github.com/ketevan614/ML-final.git
    %cd ML-final

ROOT = Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
print("IN_COLAB =", IN_COLAB, "| ROOT =", ROOT)


## Robust data directory (Fix 1)

In [ ]:
from pathlib import Path

def find_data_dir(root):
    """Locate the folder that actually holds the 5 Kaggle CSVs.

    Handles both layouts: a flat data/ folder, or the nested folder Kaggle's
    zip produces (data/walmart-recruiting-store-sales-forecasting/).
    """
    required = {"train.csv", "test.csv", "features.csv", "stores.csv", "sampleSubmission.csv"}
    for d in [root / "data", root / "data" / "walmart-recruiting-store-sales-forecasting"]:
        if d.exists() and required.issubset({p.name for p in d.glob("*.csv")}):
            return d
    raise FileNotFoundError("Could not find all 5 CSVs in data/ or the nested Kaggle folder.")

DATA_DIR = find_data_dir(ROOT)
print("DATA_DIR =", DATA_DIR)


## MLflow / DagsHub tracking (Fix 2)

In [ ]:
import os
import mlflow

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on already-exported env vars")

# stale proxy vars have previously broken the DagsHub connection in Colab
for var in ("HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"):
    os.environ.pop(var, None)

EXPERIMENT_NAME = "Prophet_Training"

def setup_tracking():
    """Use DagsHub MLflow if creds are present in the env / .env, else fall
    back to a local file-based tracking store so a bad token or a network
    hiccup never blocks a training run."""
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri and os.environ.get("MLFLOW_TRACKING_PASSWORD"):
        try:
            mlflow.set_tracking_uri(uri)
            mlflow.set_experiment(EXPERIMENT_NAME)
            print("MLflow tracking ->", uri)
            return
        except Exception as e:
            print("DagsHub tracking unavailable, falling back to local mlruns:", e)
    fallback_uri = f"file:{(ROOT / 'mlruns').as_posix()}"
    mlflow.set_tracking_uri(fallback_uri)
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("MLflow tracking -> local", fallback_uri)

setup_tracking()

def safe_log_params(params):
    try:
        mlflow.log_params(params)
    except Exception as e:
        print("mlflow.log_params failed:", e)

def safe_log_param(key, value):
    try:
        mlflow.log_param(key, value)
    except Exception as e:
        print(f"mlflow.log_param({key}) failed:", e)

def safe_log_metric(key, value):
    try:
        mlflow.log_metric(key, value)
    except Exception as e:
        print(f"mlflow.log_metric({key}) failed:", e)

def safe_log_artifact(path):
    try:
        mlflow.log_artifact(path)
    except Exception as e:
        print(f"mlflow.log_artifact({path}) failed:", e)


## Imports and data

In [ ]:
import numpy as np
import pandas as pd
import joblib

from dataloader import load_raw
from metrics import wmae
from validation import holdout_split

try:
    from prophet import Prophet
except ImportError as e:
    raise ImportError("prophet is required for this notebook (pip install prophet).") from e

train, test, features, stores, sample = load_raw(DATA_DIR)
print("train:", train.shape, "| test:", test.shape)

## Run: Prophet_SubsetSelection

Prophet is fit per series, so the brief is explicit: don't burn training time running it on
all ~3,000 (Store, Dept) pairs. Pick the `N_SUBSET` highest-volume series (by total historical
sales) - these are also the series where getting the forecast right matters most for WMAE.

In [ ]:
N_SUBSET = 5

series_totals = (
    train.groupby(["Store", "Dept"])["Weekly_Sales"].sum().sort_values(ascending=False)
)
subset_keys = series_totals.head(N_SUBSET).index.tolist()
print("selected (Store, Dept) series:")
for store, dept in subset_keys:
    print(f"  Store {store}, Dept {dept}  (total sales {series_totals[(store, dept)]:,.0f})")

with mlflow.start_run(run_name="Prophet_SubsetSelection"):
    safe_log_param("n_subset", N_SUBSET)
    safe_log_param("selection_rule", "top-N by total historical Weekly_Sales")
    safe_log_param("subset_series", str(subset_keys))

## Shared holdout + scoring helper

Single holdout split (last 39 weeks), matching the horizon used everywhere else, rather than
the full 3-fold expanding window - again to keep the per-series fit count manageable.

In [ ]:
HORIZON = 39

def series_frame(store, dept):
    s = train[(train["Store"] == store) & (train["Dept"] == dept)].sort_values("Date").reset_index(drop=True)
    return s

def fit_predict_prophet(s, holidays_df=None, yearly_seasonality=True):
    tr_mask, va_mask = holdout_split(s["Date"].to_numpy(), horizon=HORIZON)
    tr = s[tr_mask]
    va = s[va_mask]
    if len(tr) < 2 * HORIZON or len(va) == 0:
        return None, None, None

    prophet_df = tr.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"]]
    m = Prophet(
        yearly_seasonality=yearly_seasonality,
        weekly_seasonality=False,
        daily_seasonality=False,
        holidays=holidays_df,
    )
    m.fit(prophet_df)

    future = va.rename(columns={"Date": "ds"})[["ds"]]
    fc = m.predict(future)
    p = np.clip(fc["yhat"].to_numpy(), 0, None)
    y_true = va["Weekly_Sales"].to_numpy()
    hva = va["IsHoliday"].astype(bool).to_numpy()
    return wmae(y_true, p, hva), m, fc

def score_subset(holidays_df=None, yearly_seasonality=True):
    scores = []
    for store, dept in subset_keys:
        s = series_frame(store, dept)
        score, _, _ = fit_predict_prophet(s, holidays_df=holidays_df,
                                          yearly_seasonality=yearly_seasonality)
        if score is not None:
            scores.append(score)
    return scores, float(np.mean(scores)) if scores else float("nan")

## Run: Prophet_Yearly

Baseline decomposition: yearly seasonality only (additive trend + yearly seasonality, no
explicit holiday effects yet).

In [ ]:
with mlflow.start_run(run_name="Prophet_Yearly"):
    scores_yearly, mean_yearly = score_subset(holidays_df=None, yearly_seasonality=True)
    safe_log_param("yearly_seasonality", True)
    safe_log_param("holidays", "none")
    for i, s in enumerate(scores_yearly):
        safe_log_metric(f"wmae_series_{i}", s)
    safe_log_metric("wmae_subset_mean", mean_yearly)
    print("per-series WMAE:", [round(s, 1) for s in scores_yearly])
    print("subset mean WMAE (yearly only):", round(mean_yearly, 1))

## Run: Prophet_Holidays

Add the four Walmart-specific holiday weeks (Super Bowl, Labor Day, Thanksgiving, Christmas)
as a Prophet `holidays` dataframe, matching the same dates `features.py` uses for
`HOLIDAY_WEEK_BY_DATE`, then refit and compare.

In [ ]:
from features import HOLIDAY_WEEK_BY_DATE

holidays_df = pd.DataFrame(
    [(pd.Timestamp(date), name) for date, name in HOLIDAY_WEEK_BY_DATE.items()],
    columns=["ds", "holiday"],
)

with mlflow.start_run(run_name="Prophet_Holidays"):
    scores_hol, mean_hol = score_subset(holidays_df=holidays_df, yearly_seasonality=True)
    safe_log_param("yearly_seasonality", True)
    safe_log_param("holidays", "super_bowl, labor_day, thanksgiving, christmas")
    for i, s in enumerate(scores_hol):
        safe_log_metric(f"wmae_series_{i}", s)
    safe_log_metric("wmae_subset_mean", mean_hol)
    print("per-series WMAE:", [round(s, 1) for s in scores_hol])
    print("subset mean WMAE (+ holidays):", round(mean_hol, 1))

USE_HOLIDAYS = mean_hol <= mean_yearly
print("USE_HOLIDAYS =", USE_HOLIDAYS)

## Run: Prophet_Final

Refit the winning configuration on each subset series using its **full** history (no
holdout), and log the per-series models as one joblib artifact (Prophet has no native
sklearn-Pipeline-style raw->prediction wrapper the way the tree models do, so it is logged
as a plain artifact rather than `mlflow.sklearn.log_model`).

In [ ]:
final_holidays = holidays_df if USE_HOLIDAYS else None
final_models = {}
final_scores = []

for store, dept in subset_keys:
    s = series_frame(store, dept)
    score, model, _ = fit_predict_prophet(s, holidays_df=final_holidays, yearly_seasonality=True)
    if model is not None:
        final_models[(store, dept)] = model
    if score is not None:
        final_scores.append(score)

final_mean_wmae = float(np.mean(final_scores)) if final_scores else float("nan")

with mlflow.start_run(run_name="Prophet_Final"):
    safe_log_param("use_holidays", USE_HOLIDAYS)
    safe_log_param("n_subset", N_SUBSET)
    safe_log_metric("wmae_subset_mean", final_mean_wmae)
    joblib.dump(final_models, "prophet_subset_models.joblib")
    safe_log_artifact("prophet_subset_models.joblib")
    print("subset mean WMAE (final config):", round(final_mean_wmae, 1))
    print("logged", len(final_models), "per-series Prophet models as an artifact")

## Results & theory notes

- Subset holdout WMAE (yearly only): ___  | with holidays: ___
- Prophet decomposes each series additively as **trend + yearly seasonality + holiday
  effects + noise** - unlike SARIMA it doesn't need manual differencing or explicit
  ACF/PACF-driven order selection, and it degrades gracefully on short/sparse history.
- Why it doesn't replace the tree models here: it fits **one model per series** with no
  cross-series learning and no exogenous Temperature/Fuel/CPI/MarkDown signal, so a global
  model with those features (LightGBM/XGBoost) has strictly more information to work with,
  and fitting per-series models on all ~3,000 series would be very slow.
- vs SARIMA: ___ (SARIMA section covers the classical ARIMA side of this comparison)
- vs LightGBM / XGBoost on the same subset weeks: ___
